# 03 - Pipeline smoke test (designed to FINISH)

One question only: **does the pipeline work end to end** - vLLM starts, the
bundle compiles, the agent produces a patch. Nothing about throughput.

Built around two rules learned the hard way:

1. **Finish quickly, whatever happens.** Kaggle discards the log and output of a
   *cancelled* run, so a 2 GPU-hour probe that got cancelled taught us nothing.
   This is 1 task, concurrency 1, about 15 minutes, with a 30-minute hard timeout.
2. **Leave the diagnosis on disk.** Everything printed is appended to
   `/kaggle/working/smoke_log.txt`, and the runner's full stdout and stderr go to
   `smoke_stdout.txt`, so a red cell never costs us the reason.

**Please let it run to the end even if a cell goes red.**

Context is sized from the detected hardware. Anything below the grader's 32768
makes this a pipeline test only, not a comparable measurement.

In [ ]:
import glob
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

NL = chr(10)
LOG = Path('/kaggle/working/smoke_log.txt')
LOG.parent.mkdir(parents=True, exist_ok=True)


def say(msg=''):
    line = '[' + time.strftime('%H:%M:%S') + '] ' + str(msg)
    print(line, flush=True)
    with LOG.open('a', encoding='utf-8') as fh:
        fh.write(line + NL)


def nvsmi(fields):
    r = subprocess.run(['nvidia-smi', '--query-gpu=' + fields, '--format=csv,noheader'],
                       capture_output=True, text=True)
    return r.stdout.strip()


say('=== GPU ===')
info = nvsmi('name,memory.total,compute_cap')
say(info or '(nvidia-smi returned nothing)')
say('CPUs: ' + str(os.cpu_count()))

rows = [r.split(',') for r in info.splitlines() if r.strip()]
N_GPU = len(rows)
MEM_GB = sum(float(r[1].strip().split()[0]) for r in rows) / 1024 if rows else 0.0
caps = []
for r in rows:
    if len(r) > 2:
        try:
            caps.append(float(r[2].strip()))
        except ValueError:
            pass
CC = min(caps) if caps else 0.0
say('GPUs ' + str(N_GPU) + '  total ' + format(MEM_GB, '.1f') + ' GB  compute cap ' + str(CC))

assert N_GPU >= 2, 'need at least 2 GPUs, got ' + str(N_GPU)

# W4A16 compressed-tensors uses Marlin kernels, which normally need Ampere (8.0+).
# T4 is Turing (7.5). Warn and continue: whether vLLM refuses is what we are testing.
if CC and CC < 8.0:
    say('WARNING compute capability ' + str(CC) + ' is below 8.0; W4A16 Marlin')
    say('        kernels normally need Ampere or newer. vLLM may refuse.')

# 1.21 MB/token measured from the 4xL4 run; weights are ~17 GB, sharded across cards.
KV_GB = MEM_GB * 0.90 - 17.0
FITS = int(KV_GB * 1024 / 1.21) if KV_GB > 0 else 0
MAX_MODEL_LEN = 32768 if FITS >= 32768 else max(4096, (FITS // 2048) * 2048)
say('KV budget ' + format(KV_GB, '.1f') + ' GB -> about ' + format(FITS, ',') + ' tokens')
say('using max_model_len=' + str(MAX_MODEL_LEN))
assert KV_GB > 2, 'only ' + format(KV_GB, '.1f') + ' GB for KV after weights; cannot serve'
if MAX_MODEL_LEN != 32768:
    say('NOTE below the grader 32768 -> PIPELINE TEST, not a comparable measurement')

In [ ]:
for k, v in {
    'LITELLM_LOCAL_MODEL_COST_MAP': 'True',
    'TRANSFORMERS_NO_TF': '1',
    'VLLM_WORKER_MULTIPROC_METHOD': 'spawn',
    'VLLM_ENGINE_READY_TIMEOUT_S': '1200',
    'VLLM_NO_USAGE_STATS': '1',
    'OTEL_SDK_DISABLED': 'true',
    'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True',
}.items():
    os.environ[k] = v

INPUT = Path('/kaggle/input')
say('attached: ' + ', '.join(p.name for p in sorted(INPUT.iterdir())))


def find_dir(needles, must_contain=''):
    hits = []
    for p in INPUT.rglob('*'):
        if not p.is_dir():
            continue
        if any(n in p.name.lower() for n in needles):
            if not must_contain or any(p.glob(must_contain)):
                hits.append(p)
    return sorted(hits, key=lambda q: len(str(q)))[0] if hits else None


def find_model_dir(slug):
    # Kaggle mounts models at .../<instance>/<version>; matching the instance name
    # lands one level above the weights and vLLM rejects it. Anchor on config.json.
    hits = [q.parent for q in INPUT.rglob('config.json') if slug in str(q.parent).lower()]
    return sorted(hits, key=lambda q: len(str(q)))[0] if hits else None


WHEELHOUSE = find_dir(['wheelhouse'], '*.whl')
LAB = find_dir(['swegemma-lab'])
DATA = Path('/kaggle/input/competitions/gemma-4-developer-agent')
MODEL = find_model_dir('gemma-4-31b-it-qat-w4a16-ct')
for name, val in (('wheelhouse', WHEELHOUSE), ('lab', LAB), ('data', DATA), ('model', MODEL)):
    say('  ' + name.ljust(11) + str(val))
assert WHEELHOUSE and LAB and DATA.is_dir() and MODEL, 'missing an attached source'
assert (MODEL / 'config.json').is_file(), 'no config.json in ' + str(MODEL)

In [ ]:
# Decide whether this image can run the wheelhouse AT ALL, and if so with what.
#
# History, because the shape of this check changed twice:
#   - The wheelhouse ships cp312 and no cp313 wheel.
#   - Kaggle moved the GPU image to Python 3.13 on 2026-10-07, in both batch and
#     interactive mode. Pinning does not help: docker_image_pinning_type=original
#     resolves to today's image (verified -- KAGGLE_DOCKER_IMAGE matched the
#     pinned digest exactly).
#   - An earlier version of this cell asserted "interpreter tag must appear in the
#     wheelhouse" and failed the run. That was TOO BLUNT. Measured on the 3.13
#     image: vllm ships cp38-abi3 so it installs anywhere, torch is not in the
#     wheelhouse at all (it comes from the image, 2.11.0+cu128), and only 9 of 41
#     wheels are genuinely version-locked. With those 9 skipped, `import vllm`
#     succeeded and torch.cuda.is_available() was True.
#
# So the real question is not "do the tags match" but "can every version-locked
# wheel be satisfied". We vendor cp313 builds of exactly those 9, at the SAME
# versions, in the lab dataset -- a retag, not an upgrade.
import re

running = 'cp' + str(sys.version_info.major) + str(sys.version_info.minor)

def dist_name(wheel_name):
    return wheel_name.split('-')[0].lower().replace('_', '-')

def locked_tag(wheel_name):
    """The cpXY this wheel is locked to, or None if it runs on any Python.

    abi3 and pure-python wheels are not locked; 'cp313-cp313' is.
    """
    if 'abi3' in wheel_name:
        return None
    m = re.search(r'-(cp3\d+)-(cp3\d+)', wheel_name)
    return m.group(1) if m else None

OVERLAY = LAB / 'vendor' / running
overlay = {dist_name(w.name): w for w in OVERLAY.glob('*.whl')} if OVERLAY.is_dir() else {}

PLAN = []        # wheels to install, as paths
UNRESOLVED = []  # version-locked with no substitute -- these are fatal
for w in sorted(WHEELHOUSE.glob('*.whl')):
    if 'cutlass' in w.name.lower():
        continue
    tag = locked_tag(w.name)
    if tag is None or tag == running:
        PLAN.append(w)
        continue
    sub = overlay.get(dist_name(w.name))
    if sub is not None:
        PLAN.append(sub)
    else:
        UNRESOLVED.append(w.name)

tags = set()
for w in WHEELHOUSE.glob('*.whl'):
    tags.update(re.findall(r'-(cp3\d+)-', w.name))
say('running ' + running + '; wheelhouse compiled tags ' + str(sorted(tags)))
say('overlay ' + str(OVERLAY) + ': ' + str(len(overlay)) + ' wheel(s)')
say('install plan: ' + str(len(PLAN)) + ' wheel(s), ' + str(len(UNRESOLVED)) + ' unresolved')
for n in UNRESOLVED:
    say('  UNRESOLVED ' + n)

assert not UNRESOLVED, (
    'These wheels are built only for another Python and have no vendored '
    + running + ' substitute: ' + str(UNRESOLVED) + '. Download them at the SAME '
    'versions into vendor/' + running + '/ and re-run scripts/push_dataset.py. '
    'Do NOT change versions: the wheelhouse pins are what the harness expects.'
)
say('every version-locked wheel is satisfiable on ' + running)

In [ ]:
for pat in ('/usr/local/lib/python*/dist-packages/*cutlass*.pth',
            '/usr/local/lib/python*/site-packages/*cutlass*.pth'):
    for f in glob.glob(pat):
        try:
            os.unlink(f)
        except OSError:
            pass

# Install the PLAN from the previous cell, not the raw wheelhouse: the plan has
# already substituted a vendored cp313 build for each wheel the wheelhouse ships
# only as cp312.
tmp = Path('/tmp/wheelhouse')
tmp.mkdir(parents=True, exist_ok=True)
for w in PLAN:
    name = w.name
    if 'cu128' in name and '+' not in name:
        name = name.replace('cu128', '+cu128')
    if not (tmp / name).exists():
        try:
            os.symlink(w, tmp / name)
        except OSError:
            shutil.copy(w, tmp / name)

wheels = sorted(str(w) for w in tmp.glob('*.whl'))
say('installing ' + str(len(wheels)) + ' wheels')

PIP = [sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall']
r = subprocess.run(PIP + wheels, capture_output=True, text=True)
say('bulk pip rc ' + str(r.returncode))

if r.returncode != 0:
    # The official starter installs all wheels in ONE pip call, so a single
    # unsupported wheel aborts the batch and swegemma never lands. Seen on the T4
    # image: apache_tvm_ffi ... is not a supported wheel on this platform.
    say('bulk install failed; retrying per wheel')
    say((r.stderr or '')[-700:])
    failed = []
    for w in wheels:
        rr = subprocess.run(PIP + [w], capture_output=True, text=True)
        if rr.returncode != 0:
            failed.append(Path(w).name)
    say(str(len(failed)) + ' wheel(s) failed individually:')
    for f in failed:
        say('    ' + f)
    assert not failed, 'wheels failed to install: ' + str(failed)

# Prove the serving stack LINKS, not merely that pip exited 0 and the top-level
# package imports.
#
# `import vllm` is NOT sufficient, and assuming it was cost a full run on
# 2026-10-07: vllm/_C.abi3.so loads lazily, so `import vllm` printed 0.19.1
# happily and the inference server then died with
#   ImportError: vllm/_C.abi3.so: undefined symbol: _ZN3c1013MessageLoggerC1EPKciib
# That is a libtorch C++ symbol. `abi3` pins the PYTHON ABI, not the C++ one, so a
# vllm built against one torch does not link against another -- and torch is not
# in the wheelhouse, it comes from the image, which Kaggle changed underneath us.
#
# `import vllm._C` forces the native library to load, which is the cheapest check
# that reaches the real failure: seconds, against the ~2.5 minutes of model load
# and ~1 minute per task that the old check let us waste.
chk = subprocess.run(
    [sys.executable, '-c',
     'import torch;print("torch", torch.__version__, "cuda", torch.cuda.is_available());'
     'import vllm;print("vllm", vllm.__version__);'
     'import vllm._C;print("vllm._C linked OK")'],
    capture_output=True, text=True)
say('serving stack: ' + ((chk.stdout or '') + (chk.stderr or '')).strip()[-1200:])
assert chk.returncode == 0, (
    'the serving stack does not link on this image. If the error names an '
    'undefined symbol in vllm/_C.abi3.so, the wheelhouse vllm was built against a '
    'different torch than this image ships -- a C++ ABI mismatch that no cp-tag '
    'substitution can fix. See LESSONS.md 2026-10-07.')

pre = subprocess.run([sys.executable, str(LAB / 'scripts' / 'calibrate.py'), '--preflight'],
                     capture_output=True, text=True)
say('preflight:')
say((pre.stdout or '') + (pre.stderr or ''))
assert pre.returncode == 0, 'preflight failed - harness imports are broken'

In [ ]:
RUNNER = str(LAB / 'scripts' / 'session_runner.py')
OUT = Path('/kaggle/working/smoke_stdout.txt')

cmd = [sys.executable, RUNNER,
       '--lab', str(LAB), '--data', str(DATA), '--model-path', str(MODEL),
       '--variant', 'v00_public_baseline', '--task-list', 'clean_n.txt',
       '--run-id', 'smoke1', '--limit', '1', '--concurrency', '1',
       '--max-time-minutes', '6', '--max-tool-calls', '20',
       '--max-model-len', str(MAX_MODEL_LEN)]
say('running session_runner with max_model_len=' + str(MAX_MODEL_LEN))

t0 = time.time()
rc = -1
out = ''
err = ''
try:
    proc = subprocess.run(cmd, capture_output=True, text=True, timeout=30 * 60)
    rc = proc.returncode
    out = proc.stdout or ''
    err = proc.stderr or ''
except subprocess.TimeoutExpired as exc:
    rc = -9
    out = exc.stdout if isinstance(exc.stdout, str) else (exc.stdout or b'').decode('utf-8', 'replace')
    err = exc.stderr if isinstance(exc.stderr, str) else (exc.stderr or b'').decode('utf-8', 'replace')
    err = err + NL + '*** TIMEOUT at 30 min ***'
wall = time.time() - t0

parts = ['rc=' + str(rc),
         'wall=' + format(wall, '.0f') + 's',
         'max_model_len=' + str(MAX_MODEL_LEN),
         '', '=== STDOUT ===', out, '', '=== STDERR ===', err]
OUT.write_text(NL.join(parts), encoding='utf-8')
say('rc=' + str(rc) + '  wall=' + format(wall / 60, '.1f') + ' min')
say('--- stdout tail ---')
say(out[-4000:] or '(empty)')
say('--- stderr tail ---')
say(err[-4000:] or '(empty)')

In [ ]:
# Wrapped: a bug in reporting must never hide a successful run.
try:
    base = Path('/kaggle/working/results/smoke1')
    jl = base / 'task_results.jsonl'
    rows = []
    if jl.is_file():
        for line in jl.read_text(encoding='utf-8').splitlines():
            if line.strip():
                rows.append(json.loads(line))
    pdir = base / 'patches'
    patches = sorted(pdir.glob('*.patch')) if pdir.is_dir() else []
    say('=== RESULT ===')
    say('  rows            ' + str(len(rows)))
    say('  patches written ' + str(len(patches)))
    for q in patches:
        say('    ' + q.name + '  ' + str(q.stat().st_size) + ' bytes')
    for row in rows:
        say('  ' + str(row.get('instance_id')) + ': resolved=' + str(row.get('resolved'))
            + ' error=' + str(row.get('error'))[:160])
    ok = bool(patches) and patches[0].stat().st_size > 0
    say('PIPELINE WORKS' if ok else 'NO PATCH PRODUCED - read smoke_stdout.txt')
except Exception as exc:
    say('report cell raised (not fatal): ' + type(exc).__name__ + ': ' + str(exc))

In [ ]:
# Always leave something fetchable. This is the whole point of the redesign.
try:
    for d in ('results', 'ledger'):
        p = Path('/kaggle/working') / d
        if p.is_dir() and any(p.rglob('*')):
            shutil.make_archive('/kaggle/working/smoke_' + d, 'zip', root_dir=p)
            say('archived smoke_' + d + '.zip')
except Exception as exc:
    say('archive raised (not fatal): ' + str(exc))

say('=== files in /kaggle/working ===')
for q in sorted(Path('/kaggle/working').iterdir()):
    size = str(q.stat().st_size) if q.is_file() else '(dir)'
    say('  ' + q.name + '  ' + size)
say('DONE - a cancelled run discards its log, so letting this finish is what matters')